[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/en_clase/S07_01_regresion_lineal_polinomica_en_clase.ipynb)

# S07 · 01 — Regresión lineal y polinómica: predecir el precio por noche · versión en clase

**Tiempo estimado:** 40 min en clase · **Sprint 7** · M3 (UD 3.3)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/codigo_explicado/S07_01_regresion_lineal_polinomica.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** El cliente quiere una tarifa de referencia: *¿cuánto debería costar
la noche de una reserva según el tipo de habitación y la época?* Es un problema de
**regresión** (predecir un número). Empezamos por el modelo más sencillo y más explicable:
la regresión lineal.

**Al terminar sabrás:**
1. Ajustar una regresión lineal e **interpretar sus coeficientes en euros**.
2. Medir el error (MAE, RMSE, R²) y mirar los **residuos**.
3. Usar términos polinómicos y de interacción y saber cuándo compensan.
4. Comparar con un baseline y con una variable bien construida (*feature engineering*).

> 🏫 **Versión «en clase»** · ⏱ ≈ 40 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2, 3**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_01_regresion_lineal_polinomica.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)
print(df.shape)

## 1. La idea: una recta (o un plano) que se ajusta a los datos

La regresión lineal supone que el precio es una **suma ponderada** de las variables:

`precio ≈ base + (efecto de suite)·es_suite + (efecto de temporada alta)·temporada_alta + …`

Cada **coeficiente** dice *cuántos euros cambia el precio al aumentar esa variable en una
unidad, manteniendo las demás iguales*. Por eso es tan apreciada: se puede **explicar**.
Las categorías (tipo de habitación) se convierten en columnas 0/1 (*dummies*).

In [ ]:
X = pd.get_dummies(df[["tipo_habitacion"]], drop_first=True).astype(int)
X["temporada_alta"] = df["temporada_alta"]
y = df["precio_noche"]
X.head()

`drop_first=True` quita una categoría (`estandar`), que actúa como **referencia**: los
coeficientes de las otras se leen como diferencia respecto a la habitación estándar.

**Ejercicio 1 · 🏫 en clase.** Separa en `X_train, X_test, y_train, y_test` (20 % de test, `random_state=SEMILLA`),
ajusta una `LinearRegression` llamada `modelo_lin` y guarda en `r2_test` y `mae_test` sus
métricas en test.

In [ ]:
# TODO: train_test_split(X, y, test_size=0.2, random_state=SEMILLA); LinearRegression().fit(...)
...
assert len(X_test) == 1200, "Con 6000 filas y test_size=0.2 el test debe tener 1200 filas"
assert r2_test > 0.85, "R² debería superar 0,85: revisa que uses tipo de habitación y temporada"
assert 5 < mae_test < 15, "El MAE debería estar entre 5 y 15 euros"
print(f"R² = {r2_test:.3f} · MAE = {mae_test:.2f} € · RMSE = {mean_squared_error(y_test, pred) ** 0.5:.2f} €")

### ¿Mejor que "no hacer nada"?
El **baseline** de regresión predice siempre la media del precio.

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_train, y_train)
print(f"Baseline: MAE = {mean_absolute_error(y_test, base.predict(X_test)):.2f} € · R² = {r2_score(y_test, base.predict(X_test)):.3f}")

## 2. Interpretar los coeficientes

**Ejercicio 2 · 🏫 en clase.** Consulta `modelo_lin.coef_` (van en el mismo orden que las columnas de `X`).
Guarda en `coefs` un `pd.Series` con los coeficientes y el nombre de columna como índice,
y responde con números: `extra_suite` (euros extra de una suite frente a la estándar)
y `efecto_temporada_alta` (euros extra por reservar en temporada alta).

In [ ]:
# TODO: pd.Series(modelo_lin.coef_, index=X.columns)
...
assert 90 < extra_suite < 115, "Una suite cuesta unos 100 € más que una estándar (coeficiente de tipo_habitacion_suite)"
assert 15 < efecto_temporada_alta < 30, "La temporada alta añade unos 20-25 €"
print(coefs.round(1))
print("Ordenada en el origen (precio base de una estándar en temporada baja):", round(modelo_lin.intercept_, 1), "€")

## 3. Residuos: ¿el modelo se equivoca de forma "sana"?

El **residuo** es `real − predicho`. Si el modelo está bien planteado, los residuos se
reparten **sin patrón** alrededor de 0. Si vemos curvas o embudos, falta algo en el modelo.

In [ ]:
residuos = y_test - pred
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(pred, residuos, s=10, alpha=0.4, color="#2a6f97")
ejes[0].axhline(0, color="black", lw=1)
ejes[0].set(title="Residuos frente a predicción", xlabel="Precio predicho (€)", ylabel="Residuo (€)")
sns.histplot(residuos, bins=30, ax=ejes[1], color="#2a6f97")
ejes[1].set(title="Distribución de los residuos", xlabel="Residuo (€)", ylabel="Reservas")
plt.tight_layout()
plt.show()

Se ve un **embudo**: los errores crecen con el precio (la suite falla más, en euros, que la
estándar). Tiene sentido: en estos datos las variaciones de precio son **proporcionales**
(la temporada alta multiplica ×1,25 y el "ruido" es un % del precio), pero el modelo
lineal **suma** efectos fijos en euros. Una parte se puede corregir con **interacciones**
(que el efecto de la temporada dependa del tipo de habitación).

## 4. Términos polinómicos y de interacción

**Idea.** `PolynomialFeatures` crea columnas nuevas: potencias (`x²`, `x³`) y productos entre
variables (`x1·x2`, las **interacciones**). El modelo sigue siendo lineal *en los
coeficientes*, pero puede dibujar curvas. Con `interaction_only=True` solo crea productos.

In [ ]:
lineal = LinearRegression()
con_interacciones = make_pipeline(PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                                  LinearRegression())
r2_lineal_cv = cross_val_score(lineal, X, y, cv=5, scoring="r2").mean()
r2_inter_cv = cross_val_score(con_interacciones, X, y, cv=5, scoring="r2").mean()
print(f"R² CV lineal: {r2_lineal_cv:.3f} · con interacciones: {r2_inter_cv:.3f}")

**Ejercicio 3 · 🏫 en clase.** Calcula `mae_inter_test`: el MAE en test del modelo `con_interacciones`
ajustado con `X_train, y_train`. Guarda en `mejora_mae` cuántos euros mejora respecto a
`mae_test` (positivo = mejor).

In [ ]:
# TODO: ajusta con_interacciones con train y predice sobre test
...
assert mae_inter_test < mae_test, "Las interacciones deberían reducir el error"
assert mejora_mae > 0.3, "La mejora debería ser apreciable (más de 0,3 €)"

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_01_regresion_lineal_polinomica.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
